

```
Práctica N°8 - Aprendizaje Profundo
```



# **Summarization en español con Transformers**

###00. Librerías

In [1]:
!pip install -q datasets evaluate transformers[sentencepiece] rouge_score accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.7 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer
from transformers import pipeline
from huggingface_hub import notebook_login
import numpy as np
import evaluate

### 01. Login en Hugging Face

In [3]:
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### 02. Dataset

Se utiliza el dataset **Iker/NoticIA**, orientado a **summarization** en español.

In [4]:
dataset = load_dataset("Iker/NoticIA")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/132k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/231k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/50 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['web_url', 'web_headline', 'summary', 'web_text'],
        num_rows: 700
    })
    validation: Dataset({
        features: ['web_url', 'web_headline', 'summary', 'web_text'],
        num_rows: 50
    })
    test: Dataset({
        features: ['web_url', 'web_headline', 'summary', 'web_text'],
        num_rows: 100
    })
})


In [5]:
print("\nPrimer ejemplo del train:")
print(dataset["train"][0])


Primer ejemplo del train:
{'web_url': 'https://www.informacion.es/medio-ambiente/2023/11/27/jorge-rey-impactante-prediccion-tiempo-puente-diciembre-dv-95137296.html', 'web_headline': 'JORGE REY: EL TIEMPO | La impactante predicción del tiempo de Jorge Rey para el puente de diciembre', 'summary': 'El inicio de un periodo frío intenso.', 'web_text': '27·11·23 | 08:34 | Actualizado a las 14:47\nJORGE REY: EL TIEMPO | La impactante predicción del tiempo de Jorge Rey para el puente de diciembre HÉCTOR FUENTES\nEn el mundo de la meteorología, hay nombres que resuenan con autoridad y precisión. Uno de ellos es Jorge Rey, el joven burgalés que, a sus dieciséis años, ha sorprendido a España con sus predicciones climáticas. Sus métodos, basados en la observación de la naturaleza y los animales, han demostrado ser sorprendentemente precisos. Ahora, con el puente de diciembre a la vuelta de la esquina, Jorge Rey ha emitido una nueva y alarmante predicción sobre el tiempo que se espera en España. 

In [6]:
# Visualizar algunos ejemplos como tabla
dataset["train"].select(range(5)).to_pandas()

,web_url,web_headline,summary,web_text
0,https://www.informacion.es/medio-ambiente/2023...,JORGE REY: EL TIEMPO | La impactante predicció...,El inicio de un periodo frío intenso.,27·11·23 | 08:34 | Actualizado a las 14:47\nJO...
1,https://buff.ly/3Rdqz2K,El cambio en las matrículas que se espera para...,Se dará el salto a la letra M.,Si eres de los que sigues el avance de las mat...
2,https://www.cope.es/n/2978369,Si no avisas a la DGT de este cambio en tu coc...,500 euros por pintar un coche de otro color y ...,Con Pilar Cisneros y Fernando de Haro\nCon Pac...
3,https://www.genbeta.com/p/316184,Estos serán los lenguajes de programación con ...,Python y JavaScript.,Si con el año nuevo te has propuesto aumentar ...
4,https://buff.ly/47SBCap,Cambio de estrategia en Microsoft: Windows 12 ...,Solo un 28.6% de los usuarios actuales de Wind...,"Desde hace ya varios meses, las especulaciones..."


In [7]:
# Tamaño del dataset
print("Número de ejemplos en train:", len(dataset["train"]))
print("Número de ejemplos en validation:", len(dataset["validation"]))
print("Número de ejemplos en test:", len(dataset["test"]))

Número de ejemplos en train: 700
Número de ejemplos en validation: 50
Número de ejemplos en test: 100


### 03. Tokenizador

In [8]:
model_checkpoint = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

### 04. Preprocesado

En este caso se usará como entrada el titular y el texto de la noticia, y como salida el resumen humano del dataset.

In [10]:
print(dataset)
print(dataset["train"].column_names)
print(dataset["validation"].column_names)
print(dataset["test"].column_names)

DatasetDict({
    train: Dataset({
        features: ['web_url', 'web_headline', 'summary', 'web_text'],
        num_rows: 700
    })
    validation: Dataset({
        features: ['web_url', 'web_headline', 'summary', 'web_text'],
        num_rows: 50
    })
    test: Dataset({
        features: ['web_url', 'web_headline', 'summary', 'web_text'],
        num_rows: 100
    })
})
['web_url', 'web_headline', 'summary', 'web_text']
['web_url', 'web_headline', 'summary', 'web_text']
['web_url', 'web_headline', 'summary', 'web_text']


In [11]:
max_input_length = 512
max_target_length = 64

def preprocess_function(examples):

    inputs = [
        f"resume la siguiente noticia en una sola frase. Titular: {headline} Texto: {text}"
        for headline, text in zip(examples["web_headline"], examples["web_text"])
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

### 05. Data collator

In [12]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

### 06. Carga del modelo

In [13]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### 07. Métrica de evaluación

Para summarization es más apropiado usar **ROUGE**.

In [14]:
metric = evaluate.load("rouge")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {key: round(value * 100, 4) for key, value in result.items()}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = round(np.mean(prediction_lens), 4)

    return result

### 08. Argumentos de entrenamiento

In [15]:
args = Seq2SeqTrainingArguments(
    output_dir="practica8-summarization-es",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    predict_with_generate=True,
    logging_steps=50,
    load_best_model_at_end=True,
    push_to_hub=True,
    fp16=False
)

### 09. Trainer

In [16]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

### 10. Entrenamiento

In [17]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,9.231893,8.504870,10.149100,2.255400,9.102400,9.185000,19.800000
2,8.666951,8.341565,7.836700,2.063500,7.709800,7.715900,20.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=176, training_loss=8.86885573647239, metrics={'train_runtime': 2638.9073, 'train_samples_per_second': 0.531, 'train_steps_per_second': 0.067, 'total_flos': 260246706585600.0, 'train_loss': 8.86885573647239, 'epoch': 2.0})

### 11. Evaluación en test

In [18]:
results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print("Resultados finales en test:")
print(results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Resultados finales en test:
{'eval_loss': 8.318521499633789, 'eval_rouge1': 7.3869, 'eval_rouge2': 1.2398, 'eval_rougeL': 7.1475, 'eval_rougeLsum': 6.9968, 'eval_gen_len': 19.99, 'eval_runtime': 106.1089, 'eval_samples_per_second': 0.942, 'eval_steps_per_second': 0.123, 'epoch': 2.0}


### 12. Guardadar modelo

In [19]:
trainer.save_model("modelo_practica8_summarization")
tokenizer.save_pretrained("modelo_practica8_summarization")

print("Modelo guardado")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tion-es/training_args.bin: 100%|##########| 5.33kB / 5.33kB            

  ...tion-es/model.safetensors:  70%|#######   |  216MB /  308MB            

Modelo guardado


### 13. Pruebas de inferencia

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model.eval()

# Ejemplos de prueba
textos = [
    "resume la siguiente noticia en una sola frase. Titular: El Gobierno aprueba nuevas medidas educativas. Texto: El Consejo de Ministros anunció este viernes una serie de reformas orientadas a mejorar la calidad educativa y reforzar la formación del profesorado.",

    "resume la siguiente noticia en una sola frase. Titular: Hallan restos arqueológicos en una obra urbana. Texto: Durante unas obras en el centro histórico se encontraron restos arqueológicos importantes, lo que obligó a detener temporalmente los trabajos.",

    "resume la siguiente noticia en una sola frase. Titular: Aumenta el turismo en verano. Texto: Las cifras oficiales indican un incremento del turismo durante los meses de verano debido a la recuperación económica y la flexibilización de restricciones."
]

# Generación de resúmenes
for texto in textos:
    inputs = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        num_beams=4,
        do_sample=False
    )

    resumen = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\nEntrada:")
    print(texto)
    print("Resumen generado:")
    print(resumen)


Entrada:
resume la siguiente noticia en una sola frase. Titular: El Gobierno aprueba nuevas medidas educativas. Texto: El Consejo de Ministros anunció este viernes una serie de reformas orientadas a mejorar la calidad educativa y reforzar la formación del profesorado.
Resumen generado:
El Gobierno aprueba nuevas medida en una sola frase.

Entrada:
resume la siguiente noticia en una sola frase. Titular: Hallan restos arqueológicos en una obra urbana. Texto: Durante unas obras en el centro histórico se encontraron restos arqueológicos importantes, lo que obligó a detener temporalmente los trabajos.
Resumen generado:
Durante unas obras en el centro histórico se encontraron restos arqueológicos en el estado.

Entrada:
resume la siguiente noticia en una sola frase. Titular: Aumenta el turismo en verano. Texto: Las cifras oficiales indican un incremento del turismo durante los meses de verano debido a la recuperación económica y la flexibilización de restricciones.
Resumen generado:
La cifr

### 14. Subida a Hugging Face Hub

In [23]:
trainer.push_to_hub(
    commit_message="Modelo seq2seq entrenado para summarization en español con NoticIA"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tion-es/training_args.bin: 100%|##########| 5.33kB / 5.33kB            

  ...tion-es/model.safetensors:  60%|#####9    |  184MB /  308MB            

CommitInfo(commit_url='https://huggingface.co/Clau31/practica8-summarization-es/commit/85ba1f730bf9f56ee3cdc4a943258242c0e79ffd', commit_message='Modelo seq2seq entrenado para summarization en español con NoticIA', commit_description='', oid='85ba1f730bf9f56ee3cdc4a943258242c0e79ffd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Clau31/practica8-summarization-es', endpoint='https://huggingface.co', repo_type='model', repo_id='Clau31/practica8-summarization-es'), pr_revision=None, pr_num=None)